In [0]:
from pyspark.sql import functions as F


# ============================================================================
# CONFIGURATION
# ============================================================================

catalog = "workspace"
schema = "default"


# ============================================================================
# TABLE CONFIGURATION
# ============================================================================

silver_tables = [
    "silver_flights",
    "silver_aircraft",
    "silver_maintenance",
    "silver_sensor_data",
    "silver_flight_incidents",
    "silver_aerospace_flight_telemetry",
    "silver_display_telemetry",
    "silver_display_unit"
]

gold_tables = [
    "gold_flight_summary",
    "gold_aircraft_fleet_summary",
    "gold_maintenance_summary",
    "gold_sensor_summary",
    "gold_incident_summary",
    "gold_aerospace_flight_telemetry_summary",
    "gold_display_summary",
    "gold_display_telemetry_summary"
]


# ============================================================================
# SILVER DATA QUALITY CHECK
# ============================================================================

quality_results = []

for table_name in silver_tables:

    print(f"Checking Silver table: {table_name}")

    df = spark.table(
        f"{catalog}.{schema}.{table_name}"
    )

    # ------------------------------------------------------------------------
    # Total records
    # ------------------------------------------------------------------------

    total_records = df.count()


    # ------------------------------------------------------------------------
    # Duplicate records
    # ------------------------------------------------------------------------

    distinct_records = df.dropDuplicates().count()

    duplicate_records = (
        total_records - distinct_records
    )


    # ------------------------------------------------------------------------
    # Null count
    # ------------------------------------------------------------------------

    null_count_columns = [
        F.sum(
            F.when(
                F.col(column_name).isNull(),
                1
            ).otherwise(0)
        ).alias(column_name)
        for column_name in df.columns
    ]

    null_row = (
        df.select(null_count_columns)
        .collect()[0]
    )


    # ------------------------------------------------------------------------
    # Total nulls
    # ------------------------------------------------------------------------

    total_nulls = sum(
        (value or 0)
        for value in null_row
    )


    # ------------------------------------------------------------------------
    # Quality status
    # ------------------------------------------------------------------------

    if total_records > 0 and duplicate_records == 0:
        quality_status = "PASS"
    else:
        quality_status = "FAIL"


    # ------------------------------------------------------------------------
    # Store result
    # ------------------------------------------------------------------------

    quality_results.append(
        (
            table_name,
            total_records,
            duplicate_records,
            total_nulls,
            quality_status
        )
    )


# ============================================================================
# CREATE SILVER QUALITY DATAFRAME
# ============================================================================

quality_schema = [
    "table_name",
    "total_records",
    "duplicate_records",
    "total_nulls",
    "quality_status"
]


quality_df = spark.createDataFrame(
    quality_results,
    quality_schema
)


# ============================================================================
# GOLD DATA QUALITY CHECK
# ============================================================================

gold_results = []

for table_name in gold_tables:

    print(f"Checking Gold table: {table_name}")

    df = spark.table(
        f"{catalog}.{schema}.{table_name}"
    )


    # ------------------------------------------------------------------------
    # Total records
    # ------------------------------------------------------------------------

    total_records = df.count()


    # ------------------------------------------------------------------------
    # Quality status
    # ------------------------------------------------------------------------

    if total_records > 0:
        quality_status = "PASS"
    else:
        quality_status = "FAIL"


    # ------------------------------------------------------------------------
    # Store result
    # ------------------------------------------------------------------------

    gold_results.append(
        (
            table_name,
            total_records,
            quality_status
        )
    )


# ============================================================================
# CREATE GOLD QUALITY DATAFRAME
# ============================================================================

gold_schema = [
    "table_name",
    "total_records",
    "quality_status"
]


gold_quality_df = spark.createDataFrame(
    gold_results,
    gold_schema
)


# ============================================================================
# DISPLAY SILVER QUALITY
# ============================================================================

print("==============================================")
print("SILVER DATA QUALITY RESULTS")
print("==============================================")

display(
    quality_df
)


# ============================================================================
# DISPLAY GOLD QUALITY
# ============================================================================

print("==============================================")
print("GOLD DATA QUALITY RESULTS")
print("==============================================")

display(
    gold_quality_df
)


# ============================================================================
# CHECK FAILURES
# ============================================================================

silver_failures = (
    quality_df
    .filter(
        F.col("quality_status") == "FAIL"
    )
    .count()
)


gold_failures = (
    gold_quality_df
    .filter(
        F.col("quality_status") == "FAIL"
    )
    .count()
)


# ============================================================================
# FINAL JOB STATUS
# ============================================================================

if silver_failures > 0:

    raise Exception(
        f"Silver Data Quality FAILED. "
        f"Failed tables: {silver_failures}"
    )


if gold_failures > 0:

    raise Exception(
        f"Gold Data Quality FAILED. "
        f"Failed tables: {gold_failures}"
    )


print("==============================================")
print("DATA QUALITY CHECK PASSED")
print("==============================================")
print(f"Silver failures: {silver_failures}")
print(f"Gold failures: {gold_failures}")
print("==============================================")